# Make the test data

Downsample the public Jackson Laboratory endometriosis single-cell file (Tan et al. 2022) to a small h5ad per cell type. The result goes in `data/test-data/` and is the input for the `-profile test` runs.

Why downsample: NS-Forest does well when groups are close in size, and a test file must be small enough to run on a laptop. Each step below is one short code cell followed by an Explanation. Writing the steps down is the first form of the workflow; each one can later become a Nextflow module.

Method notes: Knuth 1984, Literate programming, Comput J 27:97-111. Rule et al. 2019, Ten simple rules for reproducible research in Jupyter notebooks, PLoS Comput Biol 15:e1007007.

## Step 1 - Load libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

**Explanation**

- `numpy` and `pandas` hold numbers and tables.
- `scanpy` reads and writes h5ad files.
- `scipy.sparse` holds count matrices that are mostly zeros.

## Step 2 - Get the file

The file is 8.17 GB. It is downloaded once and then reused.

In [ ]:
URL  = 'https://singlecell.jax.org/datasets/endometriosis/h5ad/endo-2022_global.h5ad'
PATH = 'data/endo-2022_global.h5ad'
os.makedirs('data', exist_ok=True)

if not os.path.exists(PATH):
    import urllib.request
    urllib.request.urlretrieve(URL, PATH)

print(f'{os.path.getsize(PATH) / 1e9:.2f} GB')

**Explanation**

- `URL` is the public address of the file at the Jackson Laboratory.
- `PATH` is where the copy is kept on your machine.
- `os.makedirs('data', exist_ok=True)` makes the folder if it is missing.
- `if not os.path.exists(PATH)` skips the download when the file is already there.
- The last line prints the file size so you can see the copy is whole.

## Step 3 - Open the file without loading it

`backed='r'` reads the file from disk as needed, so the 8 GB does not fill memory.

In [ ]:
adata = sc.read_h5ad(PATH, backed='r')
print(adata.shape)
print(list(adata.obs.columns))
print(list(adata.layers.keys()), adata.raw is not None)

**Explanation**

- `adata.shape` is the number of cells by the number of genes.
- `adata.obs.columns` lists the cell annotations the authors supplied.
- The last line shows whether raw counts sit in a layer or in `adata.raw`. limma-voom needs raw integer counts.

**Result.** Read the column names here. The next steps use `celltype_main`, `sample_type_rename` and `PID`, as `pseudobulk_limma_voom.ipynb` does. Change the names in Step 4 if they differ.

## Step 4 - Choose what to keep

In [ ]:
CELLTYPE_COL  = 'celltype_main'
CONDITION_COL = 'sample_type_rename'
DONOR_COL     = 'PID'
CONDITIONS    = ['Ctrl', 'EuE']
CELL_TYPES    = None          # set two names after reading the table below
CELLS_PER_DONOR = 50
MIN_DONORS_PER_GROUP = 3
N_GENES = 500
SEED = 1

obs = adata.obs[[CELLTYPE_COL, CONDITION_COL, DONOR_COL]]
obs = obs[obs[CONDITION_COL].isin(CONDITIONS)]
table = obs.groupby([CELLTYPE_COL, CONDITION_COL, DONOR_COL], observed=True).size()
print(table.unstack([CONDITION_COL]).fillna(0).astype(int).to_string())

**Explanation**

- `CELLTYPE_COL`, `CONDITION_COL` and `DONOR_COL` name the three annotation columns.
- `CONDITIONS` keeps eutopic endometrium from controls and from patients, the same comparison as Chapter 3.
- `CELLS_PER_DONOR` is the equal number of cells taken from each donor and cell type.
- `MIN_DONORS_PER_GROUP` is the fewest donors a cell type needs in each condition to be kept.
- `N_GENES` is how many genes the test file keeps.
- `SEED` makes the random draw the same every time.
- The table counts cells for each cell type, condition and donor.

**Result.** Pick two cell types with enough donors in both conditions, and set `CELL_TYPES` in the next cell.

## Step 5 - Draw equal-sized samples

The same number of cells is drawn from every donor within each cell type.

In [ ]:
rng = np.random.default_rng(SEED)
keep = []
for ct in CELL_TYPES:
    sub = obs[obs[CELLTYPE_COL] == ct]
    sizes = sub.groupby([CONDITION_COL, DONOR_COL], observed=True).size()
    sizes = sizes[sizes >= CELLS_PER_DONOR]
    donors_per_condition = sizes.groupby(level=0).size()
    assert (donors_per_condition >= MIN_DONORS_PER_GROUP).all() and len(donors_per_condition) == len(CONDITIONS), (ct, donors_per_condition.to_dict())
    for (cond, donor) in sizes.index:
        names = sub.index[(sub[CONDITION_COL] == cond) & (sub[DONOR_COL] == donor)]
        keep.extend(rng.choice(names, CELLS_PER_DONOR, replace=False))
print(len(keep), 'cells')

**Explanation**

- `rng` is a random number generator started from `SEED`.
- For each chosen cell type, `sizes` counts cells per condition and donor, and drops donors with fewer than `CELLS_PER_DONOR`.
- The `assert` stops the run if a cell type has too few donors in either condition.
- `rng.choice(names, CELLS_PER_DONOR, replace=False)` draws that many cells from one donor without repeats.
- `keep` collects the barcodes of all drawn cells.

## Step 6 - Cut the matrix and keep the most expressed genes

In [ ]:
sub = adata[keep].to_memory()
X = sub.raw.X if sub.raw is not None else (sub.layers['raw'] if 'raw' in sub.layers else sub.X)
if sp.issparse(X):
    X = X.tocsr()
assert np.allclose(X[:100].toarray() if sp.issparse(X) else X[:100], np.round(X[:100].toarray() if sp.issparse(X) else X[:100])), 'not integer counts'

totals = np.asarray(X.sum(axis=0)).ravel()
top = np.argsort(-totals)[:N_GENES]
genes = (sub.raw.var_names if sub.raw is not None else sub.var_names)[top]

small = sc.AnnData(X=X[:, top], obs=sub.obs[[CELLTYPE_COL, CONDITION_COL, DONOR_COL]].copy())
small.var_names = genes
small.obs['cell_type'] = small.obs[CELLTYPE_COL].astype(str)
small.obs['sample_id'] = small.obs[DONOR_COL].astype(str)
small.obs['condition'] = small.obs[CONDITION_COL].astype(str)
print(small)

**Explanation**

- `adata[keep].to_memory()` loads only the drawn cells.
- `X` takes raw counts from `adata.raw` when it exists, otherwise from `adata.X`.
- The `assert` checks the first 100 cells hold whole numbers.
- `totals` is the summed count of each gene over the drawn cells, and `top` is the positions of the `N_GENES` largest.
- `small` is a new AnnData with those genes and plain column names: `cell_type`, `sample_id` and `condition`.

## Step 7 - Write one file per cell type and a samplesheet

In [ ]:
out = 'data/test-data'
os.makedirs(out, exist_ok=True)
rows = []
for ct in CELL_TYPES:
    name = ct.replace(' ', '_').replace('/', '_')
    path = f'{out}/endometriosis_{name}.h5ad'
    small[small.obs['cell_type'] == ct].copy().write_h5ad(path, compression='gzip')
    rows.append({'cell_type': name, 'h5ad': path})
pd.DataFrame(rows).to_csv(f'{out}/bulk-de-samplesheet.csv', index=False)
for f in sorted(os.listdir(out)):
    print(f, os.path.getsize(f'{out}/{f}') // 1024, 'KB')

**Explanation**

- One h5ad is written per cell type, with `gzip` compression to keep it small.
- `bulk-de-samplesheet.csv` has the two columns `main-bulk-de.nf` reads: `cell_type` and `h5ad`.
- The loop prints each file with its size in KB.

**Result.** Copy the files in `data/test-data/` into the repository. The contrast for these files is `EuE-Ctrl`, the default in `main-bulk-de.nf`.